# CWE classification: flat vs graph traversal vs one-shot
This notebook is the experiment workflow, not a wrapper around a hidden CLI. Run cells in order (Restart Kernel and Run All). Reusable graph, scoring, artifact, and metric implementations remain in the adjacent Python modules and are unit tested. The compatibility CLI executes these same code cells.

**Default: offline synthetic smoke comparison, not a measured improvement.** No API calls, downloads, or credentials are needed for the default run. Change `demo` to false and supply an independently labeled benchmark for a real experiment. The default runs lexical flat/greedy/beam only; Chroma and Jev are opt-in. Real one-shot outputs must be supplied externally.

In [ ]:
import hashlib
import json
import math
import platform
import sys
import tempfile
from dataclasses import asdict
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

# CLI supplies its checkout root; edit the fallback for interactive use elsewhere.
ROOT = Path(globals().get('experiment_root', '/home/runner/work/cwe-classifier/cwe-classifier'))
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from benchmark import load_benchmark, metrics, paired_intervals
from classification import CWEGraph, ChromaBaseline, classify, load_scorer, normalize_label
from experiment import catalog_fingerprint, export_prompts, fingerprint, load_one_shot, write_jsonl


## 1. Freeze configuration before evaluation
Use dev data to choose margin, threshold, beam width, and Jev question formulation; do not tune on test results. Keep dev and test records in the same source file so the loader can detect text overlap across splits. `prepare` saves sanitized examples; `export-prompts` exports the full one-shot label list; `run` evaluates.

For Jev, install the `jev` extra, set `TYPESAFE_API_KEY` in the launching environment (never in cells or outputs), and select `jev_scorer:create_scorer`. See the adapter and README for the verified TypeSafe interface and model-pinning requirements. Input texts and CWE descriptions will be sent to TypeSafe only when you explicitly select that scorer; use public or approved data. Independent label questions are not candidate-relative Choice probabilities, but still require development-set calibration. Broad ancestors may get higher likelihoods than specific children, causing early stopping: inspect this bias rather than assuming the traversal stopping rule suits Jev.

No prompts are truncated. Verify the exported full prompt fits your LLM token budget; `max_prompt_chars` is only a character guard. Unknown training overlap is not evidence of a clean training split.

In [ ]:
config = dict(globals().get('experiment_config', {
    'command': 'run',
    'demo': True,
    'catalog': str(ROOT / 'data' / 'all.csv'),
    'benchmark': '/tmp/cwe-benchmark.jsonl',
    'output': '/tmp/cwe-notebook-results',
    'view': '1000',
    'split': 'test',
    'training_ids': None,
    'scorer': 'lexical',
    'methods': ['flat', 'greedy', 'beam'],
    'one_shot_predictions': None,
    'beam_width': 3,
    'top_k': 5,
    'threshold': 0.0,
    'margin': 0.0,
    'seed': 0,
    'bootstrap_samples': 2000,
    'max_prompt_chars': None,
}))
if config['command'] not in {'prepare', 'export-prompts', 'run'}:
    raise ValueError('Unknown experiment command')
if (config['beam_width'] < 1 or config['top_k'] < 1 or config['bootstrap_samples'] < 1
        or config['margin'] < 0 or not math.isfinite(config['margin'])
        or not math.isfinite(config['threshold'])):
    raise ValueError('Invalid traversal or bootstrap parameters')
if not config['methods'] or not set(config['methods']) <= {'flat', 'greedy', 'beam', 'chroma', 'one-shot'}:
    raise ValueError('Choose supported experiment methods')
if config.get('demo') and (config['scorer'] != 'lexical' or set(config['methods']) - {'flat', 'greedy', 'beam'}):
    raise ValueError('Demo is offline lexical only; disable demo for real backends')
print(json.dumps(config, indent=2))


## 2. Inspect the catalog and exclusion audit
The selected view filters ChildOf edges, not the whole catalog. Multiple parents and disconnected nodes are retained; obsolete/deprecated entries are excluded. JSONL needs explicit dev/test splits; an offline NVD dump defaults to test. Sanitization removes CWE identifiers and deduplication excludes cross-split overlaps before selection. Gold labels are acceptable alternatives; unsupported alternatives are retained when at least one gold label is eligible.

The three demo examples below are handcrafted pipeline checks, not a held-out vulnerability dataset. Save the displayed original input hash and audit separately when using `prepare` or `export-prompts`; only `run` writes these in summary.json. For publication, evaluate the original combined benchmark, not just a prefiltered subset.

In [ ]:
graph = CWEGraph(config['catalog'], config['view'])
training_ids = None
if config.get('training_ids'):
    training_ids = json.loads(Path(config['training_ids']).read_text(encoding='utf-8'))
    if not isinstance(training_ids, list) or not all(isinstance(x, str) for x in training_ids):
        raise ValueError('training_ids must contain a JSON array of strings')
demo_directory = None
benchmark_path = Path(config['benchmark'])
if config.get('demo'):
    demo_directory = tempfile.TemporaryDirectory(prefix='cwe-notebook-')
    benchmark_path = Path(demo_directory.name) / 'demo.jsonl'
    write_jsonl(benchmark_path, [
        {'id': 'synthetic-sql', 'text': 'User input is concatenated into a SQL query and executed by the database.', 'labels': ['89'], 'split': 'test'},
        {'id': 'synthetic-xss', 'text': 'Untrusted text is rendered as HTML and runs script in a browser.', 'labels': ['79'], 'split': 'test'},
        {'id': 'synthetic-bounds', 'text': 'A memory write exceeds the allocated buffer boundary.', 'labels': ['787', '119'], 'split': 'test'},
    ])
raw_benchmark_hash = hashlib.sha256(benchmark_path.read_bytes()).hexdigest()
examples, audit = load_benchmark(benchmark_path, list(graph.labels), config['split'], training_ids)
for example in examples:
    example['labels'] = [normalize_label(label) for label in example['labels']]
if demo_directory is not None:
    demo_directory.cleanup()
if not examples:
    raise ValueError('No eligible examples remain; inspect benchmark and split')
print('Catalog labels:', len(graph.labels), 'Roots:', len(graph.roots), 'Examples:', len(examples))
print(json.dumps(audit, indent=2))
print('Original input SHA-256:', raw_benchmark_hash)
print(json.dumps(examples[:3], indent=2))


## 3. Optional benchmark preparation and one-shot export
Set `command` to `export-prompts` and `output` to a JSONL file. Use an approved model runner/background agents to read the exported prompts and return real single-call outputs without retrieval, few-shot examples, traversal, or retries. Do not script Copilot CLI calls. Preserve id/text_hash/catalog_hash; record model, calls=1, truncated_labels=[], predictions (0 or 1 CWE), latency_seconds, and optional input_tokens/output_tokens/cost. Never fabricate outputs to fill this baseline. Switch back to `run`, include `one-shot` in methods, and point one_shot_predictions to the output JSONL.

In [ ]:
if config['command'] == 'prepare':
    write_jsonl(config['output'], examples)
    print('Saved sanitized benchmark:', config['output'])
elif config['command'] == 'export-prompts':
    export_prompts(examples, graph, config['output'], config['max_prompt_chars'])
    print('Saved complete-label one-shot prompts:', config['output'])


## 4. Matched comparison: same text and eligible labels
Flat scores all labels. Greedy keeps one route; beam retains multiple active routes and deduplicates shared DAG nodes. Both stop unless a child improves on its parent by more than margin and meets threshold. Caching and score comparability are necessary: candidate-relative Choice distributions cannot be compared between a parent's and child's question sets. Beam search may reduce early routing mistakes; this is a hypothesis, not a result.

Chroma uses the same descriptions/embedding approach as the original search, but a fresh ephemeral index and matched eligible catalog. Index initialization time is excluded from query latency. Selecting Chroma can download an embedding model. Scorer calls count adapter invocations; Jev's actual request counts and model metadata are separately captured when available.

In [ ]:
results = {}
scorer_metadata = {}
if config['command'] == 'run':
    if 'one-shot' in config['methods']:
        if not config.get('one_shot_predictions'):
            raise ValueError('Export prompts and supply genuine one-shot predictions first')
        results['one-shot'] = load_one_shot(config['one_shot_predictions'], examples, graph)
    scorer = load_scorer(config['scorer'], graph.labels) if set(config['methods']) & {'flat', 'greedy', 'beam'} else None
    chroma = None
    try:
        chroma = ChromaBaseline(graph.labels) if 'chroma' in config['methods'] else None
        scorer_metadata = dict(getattr(scorer, 'metadata', {}))
        for method in dict.fromkeys(config['methods']):
            if method == 'one-shot':
                continue
            records = []
            for example in examples:
                requests_before = getattr(scorer, 'requests', None)
                if method == 'chroma':
                    prediction = chroma.predict(example['text'], config['top_k'])
                else:
                    width = {'flat': None, 'greedy': 1, 'beam': config['beam_width']}[method]
                    prediction = classify(example['text'], graph, scorer, width, config['top_k'], config['threshold'], config['margin'])
                record = {**example, **asdict(prediction)}
                if method != 'chroma' and requests_before is not None:
                    record['model_requests'] = scorer.requests - requests_before
                records.append(record)
            results[method] = records
            print(method, 'completed:', len(records), 'examples')
    finally:
        close_scorer = getattr(scorer, 'close', None)
        if callable(close_scorer):
            close_scorer()
    scorer_metadata = dict(getattr(scorer, 'metadata', {}))


## 5. Metrics, paired intervals, and reproducibility artifacts
Top-1 accuracy accepts any annotated gold label. Macro-F1 uses acceptable-target top-1 resolution (not standard multilabel F1); the exact convention is recorded in the output. Top-k recall is the fraction of examples with any acceptable hit; compare methods at the same k. Distance is minimum undirected top-1-to-gold graph distance, excluding abstentions and separately counting disconnected predictions. Usage totals are null when any record is unknown; known subtotals do not imply a zero-cost run.

Paired percentile-bootstrap 95% intervals resample matched examples, using the configured seed. They do not correct training contamination or correlated duplicate vulnerabilities. Preserve the input data, lockfile, model revision, and settings when sharing results. Pin Jev rather than a moving model alias. The notebook records installed package versions and hashes but cannot prove remote-model determinism.

In [ ]:
summary = None
if config['command'] == 'run':
    packages = {}
    for package in ['chromadb', 'typesafe-sdk', 'jupyterlab', 'nbformat', 'nbclient']:
        try:
            packages[package] = version(package)
        except PackageNotFoundError:
            packages[package] = None
    summary = {
        'configuration': config,
        'synthetic_demo': bool(config.get('demo')),
        'catalog_hash': catalog_fingerprint(graph),
        'benchmark_hash': fingerprint(examples),
        'raw_benchmark_hash': raw_benchmark_hash,
        'workflow_hash': hashlib.sha256((ROOT / 'experiments.ipynb').read_bytes()).hexdigest(),
        'audit': audit,
        'training_overlap': 'known IDs excluded' if training_ids is not None else 'unknown',
        'environment': {'python': platform.python_version(), 'packages': packages, 'lock_hash': hashlib.sha256((ROOT / 'uv.lock').read_bytes()).hexdigest()},
        'scorer_metadata': scorer_metadata,
        'metrics': {method: metrics(records, graph) for method, records in results.items()},
        'paired_accuracy_intervals': paired_intervals(results, config['seed'], config['bootstrap_samples']) if len(results) > 1 else {},
        'notes': ['Synthetic/lexical checks are not evidence about learned classification.', 'Tune on dev only; freeze settings before test.', 'One-shot supplies top-1 only.', 'Setup/indexing time is excluded; unknown usage remains null.'],
    }
    output = Path(config['output'])
    output.mkdir(parents=True, exist_ok=True)
    for method, records in results.items():
        write_jsonl(output / f'{method}.jsonl', records)
    (output / 'summary.json').write_text(json.dumps(summary, indent=2, sort_keys=True), encoding='utf-8')
    print(json.dumps(summary['metrics'], indent=2))
    print(json.dumps(summary['paired_accuracy_intervals'], indent=2))
    print('Artifacts:', output)


## 6. Inspect early routing errors
Look for examples where flat predicts an acceptable CWE but greedy does not, and whether beam recovers it. Paths reveal which branch was selected. These diagnostics are descriptive, not an invitation to retune on the test set. Do not publish notebook outputs containing private input texts, credentials, or sensitive paths.

In [ ]:
routing_errors = []
if config['command'] == 'run' and {'flat', 'greedy'} <= results.keys():
    for index, flat in enumerate(results['flat']):
        greedy = results['greedy'][index]
        flat_correct = bool(flat['predictions']) and flat['predictions'][0] in flat['labels']
        greedy_correct = bool(greedy['predictions']) and greedy['predictions'][0] in greedy['labels']
        if flat_correct and not greedy_correct:
            beam = results.get('beam', [])[index] if 'beam' in results else None
            routing_errors.append({'id': flat['id'], 'gold': flat['labels'], 'flat': flat['predictions'], 'greedy_paths': greedy['paths'], 'beam_paths': beam['paths'] if beam else None})
    print(json.dumps(routing_errors[:10], indent=2))
